<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [1]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [3]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

Here's the cell filled in with what actually happened, excerpted:

*Weak: "add a search feature"*

> *"Sure! I'll add a search feature. Here's a plan: create a new `search.py` module, install `whoosh` or `elasticsearch-py`... build an index on startup... Want me to go ahead and implement this?"*

*Project-aware: "read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

> *"Add `tags: list[str] | None = None` to `search_documents`'s signature — typed per AGENTS.md's 'type every public signature.' Validate at the boundary... Filter before retrieval: narrow `documents` to `[d for d in documents if not tags or set(d.tags) & set(tags)]`, then pass that subset into `retrieve(...)`... Update the tool's `description` string... No changes to `get_document_metadata`, `summarize_document`, `llm.py`... No new dependency."*

*Differences noted:*
- The weak prompt didn't check whether a search feature already existed — it invented a whole new module and a new dependency for something that was three lines from being extended correctly. `search_documents` was already there.
- The project-aware prompt only proposed touching the one function/file it was asked about, and named the exact lines it would change.
- The weak prompt jumped straight to "want me to implement this?" with no plan-review step. The project-aware prompt stayed plan-only, as instructed.
- The project-aware prompt reused something that already existed in the codebase (`Document.tags`, already used in `get_document_metadata`) rather than inventing a new concept — it only found this because it actually read `tools.py` first, not just `AGENTS.md`.
- No new dependency in the project-aware plan; the weak one added one by default.

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** The tags-matching semantics — I used "match if the document has any of the requested tags" (wanted & set(doc.tags), an OR). An equally defensible choice would be "match only if it has all of them" (AND). Neither is stated in the exercise. Is silently picking OR semantics the kind of unstated assumption you'd want to catch and challenge?
The pool = documents fallback line — technically unnecessary since tags defaults falsy and the if tags: branch just wouldn't reassign pool; I could've written pool = documents if not tags else [...] in one line instead of two. Minor, but it's exactly the kind of "did the assistant write more than needed" question step 3 wants you to ask.

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

*Where the assistant assumed wrong:* in step 2, I planned and implemented "match if a document has any of the requested tags" (OR semantics) without ever raising it as a decision — it went straight from plan to code. The "state the plan" instruction in AGENTS.md already existed, but nothing required flagging which of several valid interpretations was chosen when a parameter is genuinely ambiguous. That's a real gap, not just a style nit: two learners running the same prompt could get OR or AND behavior with no signal either was a choice.

Edit made (on the scratch/tags-filter branch, committed nowhere yet — this is AGENTS.md, not the notebook, so it's a real project-policy change, not a TODO(you) fill-in): added a rule under Coding rules requiring the plan step to state matching/combination semantics explicitly whenever more than one reasonable interpretation exists, rather than letting the learner discover the choice by reading the diff after the fact.

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [7]:
loop = {
    "plan_approved": (
        "Add optional tags param to search_documents only; filter to docs "
        "sharing any tag (OR) before retrieve."
    ),
    "diff_inspected": (
        "git diff tools.py: tags param, filter block before retrieve(), "
        "description string. Nothing else touched."
    ),
    "rejected_change": (
        "Early return when the filtered pool was empty, before calling retrieve."
    ),
    "why_rejected": (
        "Dead code — retrieve() on an empty pool already returns [], and the "
        "existing no-results guard already handles it."
    ),
    "risks": (
        "OR vs AND tag semantics is unvalidated; no regression test since "
        "tests/ is withheld."
    ),
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      Add optional tags param to search_documents only; filter t
diff_inspected     git diff tools.py: tags param, filter block before retriev
rejected_change    Early return when the filtered pool was empty, before call
why_rejected       Dead code — retrieve() on an empty pool already returns []
risks              OR vs AND tag semantics is unvalidated; no regression test


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [8]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [9]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.